# Lesson 11 Lab — Fused Softmax

**Puzzle:** When row programs, stable exponentials, and one-pass traffic change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates row programs, stable exponentials, and one-pass traffic and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A fused row Softmax loads a row, subtracts its maximum, exponentiates, reduces the denominator, normalizes, and stores once. The power-of-two block is an internal padded shape; masks keep the logical column count exact.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["row programs, stable exponentials, and one-pass traffic"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A one-row program can exceed practical register capacity for very wide rows; a multi-stage design may then be required.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 11
LESSON_TITLE = 'Fused Softmax'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260824
}


## 5. Freeze the experiment

**Experiment:** Compare a stable Triton row Softmax with torch.softmax on 4,096 by 1,024 FP32 values.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.020527999848127365,
  "secondary": 0.012511999811977148,
  "max_abs_error": 1.4901161193847656e-08,
  "passed": true,
  "details": {
    "triton_samples_ms": [
      0.03411199897527695,
      0.0261439997702837,
      0.023360000923275948,
      0.021503999829292297,
      0.0208320003002882,
      0.025567999109625816,
      0.02127999998629093,
      0.020576000213623047,
      0.02051199972629547,
      0.020447999238967896,
      0.01961600035429001,
      0.01926399953663349,
      0.019200000911951065,
      0.01958400011062622,
      0.019007999449968338,
      0.01817600056529045,
      0.02035200037062168,
      0.0208320003002882,
      0.02054399996995926,
      0.01897599920630455
    ],
    "pytorch_samples_ms": [
      0.013055999763309956,
      0.012992000207304955,
      0.012543999589979649,
      0.012032000347971916,
      0.01196799986064434,
      0.011935999616980553,
      0.012384000234305859,
      0.012223999947309494,
      0.01219199970364

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton median | 0.0205 ms |
| torch.softmax median | 0.0125 ms |
| Maximum absolute error | 1.490e-08 |
| Acceptance gate | true |


## 8. Explain without overclaiming

One Triton program per row fused max, exp, sum, and normalization in 0.0205 ms versus 0.0125 ms for torch.softmax; max error was 1.49e-08.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use the fused kernel only inside its validated row-width, dtype, and error envelope.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 11,
  "title": "Fused Softmax",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260824
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.020527999848127365,
    "secondary": 0.012511999811977148,
    "max_abs_error": 1.4901161193847656e-08,
    "passed": true,
    "details": {
      "triton_samples_ms": [
        0.03411199897527695,
        0.0261439997702837,
        0.023360000923275948,
        0.021503999829292297,
        0.0208320003002882,
        0.025567999109625816,
        0.02127999998629093,
        0.020576000213623047,
        0.02051199972629547,
        0.020447999238967896,
        0.01961600035429001,
        0.01926399953663349,
        0.019200000911951065,
        0.01958400011062622,
        0

## 10. Make the bounded decision

> Use the fused kernel only inside its validated row-width, dtype, and error envelope.

**Failure analysis:** A one-row program can exceed practical register capacity for very wide rows; a multi-stage design may then be required.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
